In [1]:
import numpy as np
from pathlib import Path

root = Path.cwd().parents[1] if Path.cwd().name == "reencoding" else Path.cwd()

with np.load(
    root / "environment/task_inputs/train.npz",
    allow_pickle=False
) as data:
    X = data["X"][:1000].copy()
    y = data["y"][:1000].copy()
    state_ids = data["state_ids"][:1000].copy()
    features = data["feature_names"].tolist()

assert np.all(state_ids == "CA")
X_original = X.copy()
y_original = y.copy()
state_ids_original = state_ids.copy()
assert np.all(state_ids == "CA")
X_original = X.copy()
y_original = y.copy()
state_ids_original = state_ids.copy()
print(X.shape, y.shape, state_ids.shape)
print(features)
X_original = X.copy()
y_original = y.copy()
state_ids_original = state_ids.copy()


(1000, 10) (1000,) (1000,)
['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'RAC1P']


In [2]:
example_X = np.array([35, 42, 29])
example_y = np.array([1, 0, 0])
order = np.array([2, 0, 1])
print("Ages:", example_X[order])
print("Labels:", example_y[order])

Ages: [29 35 42]
Labels: [0 1 0]


## Joint-shuffle check

Apply one seeded permutation to all three arrays and invert it to check row alignment.

In [4]:
rng = np.random.default_rng(42)
order = rng.permutation(len(y))
X_shuffled, y_shuffled, state_ids_shuffled = X[order], y[order], state_ids[order]
inverse_order = np.argsort(order)
X_restored, y_restored, states_restored = X_shuffled[inverse_order], y_shuffled[inverse_order], state_ids_shuffled[inverse_order]
misaligned = np.any(X_restored != X, axis=1) | (y_restored != y) | (states_restored != state_ids)
assert not misaligned.any()
print("Joint-shuffle misaligned rows:", int(misaligned.sum()))

Joint-shuffle misaligned rows: 0


## Numerical baselines and candidates

2018 ACS PUMS codes AGEP 0 for under one year and 1–99 years (top-coded). WKHP is 1–98 hours, with 99 meaning 99 or more. Following ACSIncome filtering, this sample’s encoded valid ranges are 17–99 and 1–99. [2018 ACS PUMS Data Dictionary](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2018.pdf)

The candidate jitter table records bounds, distortion, equality, and rank preservation. Values at the upper top-code are held fixed.

In [6]:
for name in ["AGEP", "WKHP"]:
    values = X[:, features.index(name)]
    print(name, "min:", values.min(), "max:", values.max(), "mean:", round(values.mean(), 2), "std:", round(values.std(), 2))

AGEP min: 17.0 max: 86.0 mean: 27.53 std: 13.24
WKHP min: 1.0 max: 99.0 mean: 33.09 std: 19.48


In [7]:
import pandas as pd
from scipy.stats import spearmanr
magnitudes = [0.1, 0.25, 0.5, 1.0]
bounds = {"AGEP": (17.0, 99.0), "WKHP": (1.0, 99.0)}
rng = np.random.default_rng(2026)
jitter_candidates, results = {}, []
for name in ["AGEP", "WKHP"]:
    col = features.index(name)
    original = X[:, col].copy()
    lower, upper = bounds[name]
    eligible = original != 99.0
    for magnitude in magnitudes:
        proposed = original.copy()
        proposed[eligible] += rng.uniform(-magnitude, magnitude, int(eligible.sum()))
        outside = (proposed < lower) | (proposed > upper)
        transformed = np.clip(proposed, lower, upper)
        jitter_candidates[(name, magnitude)] = transformed.copy()
        change = np.abs(transformed - original)
        results.append({"feature":name,"half_width":magnitude,"outside_before":int(outside.sum()),"outside_after":int(((transformed<lower)|(transformed>upper)).sum()),"min_after":transformed.min(),"max_after":transformed.max(),"mean_abs":change.mean(),"max_abs":change.max(),"mean_delta":transformed.mean()-original.mean(),"std_delta":transformed.std()-original.std(),"exact_pct":100*np.mean(transformed==original),"unique_before":np.unique(original).size,"unique_after":np.unique(transformed).size,"spearman":spearmanr(original,transformed).statistic})
print(pd.DataFrame(results).round(4).to_string(index=False))
assert np.array_equal(X, X_original) and np.array_equal(y, y_original) and np.array_equal(state_ids, state_ids_original)
print("Original sample unchanged.")

feature  half_width  outside_before  outside_after  min_after  max_after  mean_abs  max_abs  mean_delta  std_delta  exact_pct  unique_before  unique_after  spearman
   AGEP        0.10               6              0    17.0000    86.0099    0.0491   0.1000      0.0046     0.0035        0.6             60           995    0.9955
   AGEP        0.25               5              0    17.0000    86.1866    0.1263   0.2497     -0.0026    -0.0057        0.5             60           996    0.9955
   AGEP        0.50               4              0    17.0000    86.4476    0.2485   0.5000     -0.0060     0.0064        0.4             60           997    0.9955
   AGEP        1.00               7              0    17.0000    85.5159    0.4804   0.9998      0.0013     0.0191        0.7             60           994    0.9856
   WKHP        0.10               0              0     1.0121    99.0000    0.0488   0.0999      0.0032     0.0013        1.2             68           989    0.9916
   WKHP   

## Categorical recoding experiment

Create one reversible feature-specific mapping from the full numeric code sets in the local 2018 and 2021 PUMS dictionaries. The same mapping is reused across every state and year. `RELP` is first harmonized to the documented common categories (including pooling 2018 roomer/boarder into other nonrelative). Missing values use a separate sentinel; unrecognized codes raise an error. `SCHL` receives opaque category codes, with its ordinal rank retained in a separate in-memory sidecar.

This bijection is for representation testing, not a privacy guarantee: an internal crosswalk recovers every source category. Census codebooks: [2018](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2018.pdf) and [2021](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2021.pdf).

In [9]:
import re
import sys
from folktables import ACSDataSource
from folktables.acs import adult_filter
sys.path.insert(0, str(root))
from scripts.package_data import DATA_DIR, harmonize_features

categorical_features = ["COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "SEX", "RAC1P"]
rel_map = {20:0,21:1,22:13,23:1,24:13,25:2,26:3,27:4,28:5,29:6,30:7,31:8,32:9,33:10,34:12,35:14,36:15,37:16,38:17}
codebooks, meanings = {}, {}
for year in (2018, 2021):
    dictionary = pd.read_csv(root / "exploration/eda_outputs" / f"official_definitions_{year}.csv", header=None, dtype=str).fillna("")
    codebooks[year], meanings[year] = {}, {}
    for feature in categorical_features:
        source_feature = "RELP" if feature == "RELP" and year == 2018 else "RELSHIPP" if feature == "RELP" else feature
        entries = dictionary[(dictionary[0] == "VAL") & (dictionary[1] == source_feature)]
        codes, labels = {}, {}
        for _, row in entries.iterrows():
            raw_code = row[4].strip()
            if not re.fullmatch(r"\d+", raw_code):
                continue
            code_value = int(raw_code)
            if feature == "RELP":
                code_value = ({11:15}.get(code_value, code_value) if year == 2018 else rel_map[code_value])
            codes[code_value] = row[6].strip()
        codebooks[year][feature] = set(codes)
        meanings[year][feature] = codes

seed = np.random.default_rng(1403)
category_maps, inverse_maps, schl_order = {}, {}, {}
codebook_rows = []
for feature in categorical_features:
    codes18, codes21 = codebooks[2018][feature], codebooks[2021][feature]
    if codes18 != codes21:
        raise ValueError(f"Category code sets differ across years for {feature}")
    codes = sorted(codes18)
    opaque = seed.choice(np.arange(10000, 99999), size=len(codes), replace=False)
    category_maps[feature] = dict(zip(codes, map(int, opaque)))
    inverse_maps[feature] = {token: code for code, token in category_maps[feature].items()}
    if feature == "SCHL":
        schl_order = {code: rank for rank, code in enumerate(codes)}
    if feature == "RELP":
        label_differences = "intentional RELP/RELSHIPP pooling"
    else:
        label_changes = [
            code for code in codes
            if re.sub(r"[^a-z0-9]+", " ", meanings[2018][feature][code].lower().replace("&", "and")).strip()
            != re.sub(r"[^a-z0-9]+", " ", meanings[2021][feature][code].lower().replace("&", "and")).strip()
        ]
        label_differences = len(label_changes)
    codebook_rows.append({"feature":feature,"codes_per_year":len(codes),"same_code_set":True,"meaning_label_differences":label_differences})
    if feature != "RELP" and label_differences:
        print(f"{feature} codebook descriptions needing semantic review:", label_changes)

MISSING_TOKEN = -1

def encode_categories(array):
    encoded = array.copy()
    for feature in categorical_features:
        col = features.index(feature)
        mapping = category_maps[feature]
        vals = []
        for value in array[:, col]:
            if np.isnan(value) or (value == 0 and 0 not in mapping):
                vals.append(MISSING_TOKEN)
            elif float(value).is_integer() and int(value) in mapping:
                vals.append(mapping[int(value)])
            else:
                raise ValueError(f"Unseen {feature} category: {value}")
        encoded[:, col] = vals
    return encoded

def decode_categories(array):
    decoded = array.copy()
    for feature in categorical_features:
        col = features.index(feature)
        inverse = inverse_maps[feature]
        vals = []
        for value in array[:, col]:
            vals.append((0 if 0 not in inverse.values() else np.nan) if value == MISSING_TOKEN else inverse[int(value)])
        decoded[:, col] = vals
    return decoded

X_categorical = encode_categories(X)
X_recovered = decode_categories(X_categorical)
assert np.array_equal(X_recovered, X, equal_nan=True)
schl_col = features.index("SCHL")
schl_order_values = np.array([schl_order[int(v)] for v in X[:, schl_col]])
ordered_schl_codes = sorted(schl_order)
assert all(schl_order[a] < schl_order[b] for i, a in enumerate(ordered_schl_codes) for b in ordered_schl_codes[i + 1:])
assert all(len(set(m.values())) == len(m) for m in category_maps.values())
assert all(inverse_maps[f][token] == code for f, m in category_maps.items() for code, token in m.items())

missing_probe = X[:1].copy()
missing_probe[0, features.index("COW")] = np.nan
assert encode_categories(missing_probe)[0, features.index("COW")] == MISSING_TOKEN
zero_missing_probe = X[:1].copy()
zero_missing_probe[0, features.index("COW")] = 0
assert encode_categories(zero_missing_probe)[0, features.index("COW")] == MISSING_TOKEN
relp_zero_probe = X[:1].copy()
relp_zero_probe[0, features.index("RELP")] = 0
assert encode_categories(relp_zero_probe)[0, features.index("RELP")] == category_maps["RELP"][0]
unknown_probe = X[:1].copy()
unknown_probe[0, features.index("COW")] = -987654
try:
    encode_categories(unknown_probe)
except ValueError:
    unknown_rejected = True
else:
    unknown_rejected = False
assert unknown_rejected
print(pd.DataFrame(codebook_rows).to_string(index=False))
print("Sample categories mapped:", sum(len(set(X[:, features.index(f)])) for f in categorical_features))
print("Round-trip recovery: exact; NaN/zero missing: handled; valid RELP=0 preserved; unseen code: rejected")
print("SCHL ordinal sidecar rows:", len(schl_order_values), "distinct ranks:", len(np.unique(schl_order_values)))

OCCP codebook descriptions needing semantic review: [1555, 2636, 2810, 2920, 3720, 3725, 3930, 5740, 6700, 6800, 7120, 8000, 9350, 9920]
feature  codes_per_year  same_code_set         meaning_label_differences
    COW               9           True                                 0
   SCHL              24           True                                 0
    MAR               5           True                                 0
   OCCP             530           True                                14
   POBP             224           True                                 0
   RELP              17           True intentional RELP/RELSHIPP pooling
    SEX               2           True                                 0
  RAC1P               9           True                                 0
Sample categories mapped: 333
Round-trip recovery: exact; NaN/zero missing: handled; valid RELP=0 preserved; unseen code: rejected
SCHL ordinal sidecar rows: 1000 distinct ranks: 16


## OCCP crosswalk review

The Census PUMS `definition.csv` files cached with the 2018 and 2021 releases both define OCCP using 2018 occupation codes. Their OCCP value labels are compared with the Census 2018 occupation list/crosswalk. This review compares the 14 changed year-specific labels to the official 2018 ACS PUMS OCCP value labels and the Census 2018 occupation list/crosswalk. The Census report documents title updates such as `correspondents` to `journalists` and `motion picture` to `film`; those are title changes, not evidence of a recode. Most differences are therefore label wording/abbreviation changes. Code 6800 remains unresolved because the dictionaries describe it as including roustabouts (and the 2018 label also says mining), while the 2018 Census list separates roustabouts as 6920 and defines 6800 as oil-and-gas operators.

Sources: [2018 ACS PUMS Data Dictionary](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2018.pdf), [Census 2018 occupation code lists and crosswalks](https://www.census.gov/topics/employment/industry-occupation/guidance/code-lists.html), [Census classification changes report, Appendix A-2](https://www.census.gov/content/dam/Census/library/publications/2020/demo/acs-tp78.pdf).


In [11]:
# Descriptions are read from the Census-hosted year-specific OCCP definitions above.
normalized_label = lambda x: re.sub(r"[^a-z0-9]+", " ", x.lower().replace("&", "and")).strip()
# Read the source definition.csv files bundled with each Census PUMS download.
def source_occp_labels(year):
    import csv
    path = root / "exploration" / "data" / str(year) / "1-Year" / "definition.csv"
    labels = {}
    with path.open(newline="") as file:
        for row in csv.reader(file):
            if len(row) >= 7 and row[0] == "VAL" and row[1] == "OCCP":
                if re.fullmatch(r"\d+", row[4].strip()):
                    labels[int(row[4])] = row[6].strip()
    return labels

canonical_2018 = source_occp_labels(2018)
source_2021 = source_occp_labels(2021)
assert canonical_2018 == meanings[2018]["OCCP"]
assert source_2021 == meanings[2021]["OCCP"]
print("Canonical OCCP labels loaded from cached Census definition.csv for 2018; both year dictionaries validated.")
occ_diff_codes = [
    code for code in sorted(codebooks[2018]["OCCP"] & codebooks[2021]["OCCP"])
    if normalized_label(meanings[2018]["OCCP"][code])
    != normalized_label(meanings[2021]["OCCP"][code])
]

occ_assessment = {
    1555: ("Wording only", "Category title/prefix formatting; same 2018 OCCP code."),
    2636: ("Wording only", "Pluralization in 'window(s) trimmers'; canonical title is window trimmers."),
    2810: ("Wording only", "Census report documents title update from correspondents to journalists; code 2810 retained."),
    2920: ("Wording only", "Census report documents motion picture to film title update; code 2920 retained."),
    3720: ("Wording only", "Fire fighting/firefighting wording; same canonical code group."),
    3725: ("Wording only", "PUMS residual-group title was modernized; both years use the 2018 OCCP coding vintage."),
    3930: ("Wording only", "Gaming/gambling wording; canonical 2018 title uses gambling."),
    5740: ("Wording only", "2018 'Medial' is a documentation typo; canonical title is Medical."),
    6700: ("Wording only", "2018 label omits escalator; canonical 2018 code title includes elevator and escalator installers."),
    6800: ("Unresolved", "PUMS labels include roustabouts; 2018 Census list assigns roustabouts to 6920 and 6800 to oil/gas operators."),
    7120: ("Wording only", "Electronic home-entertainment wording updated to the 2018 SOC title audiovisual equipment installers."),
    8000: ("Wording only", "2018 label is abbreviated; 2021 spells out setters, operators, and tenders."),
    9350: ("Wording only", "Parking attendants vs parking lot attendants; same canonical group."),
    9920: ("Wording only", "Equivalent phrasing for the same unemployed/no-recent-work category."),
}
assert set(occ_diff_codes) == set(occ_assessment), (occ_diff_codes, sorted(occ_assessment))
occ_review = pd.DataFrame([
    {"OCCP": c,
     "2018 description": meanings[2018]["OCCP"][c],
     "2018 canonical PUMS label": meanings[2018]["OCCP"][c],
     "2021 description": meanings[2021]["OCCP"][c],
     "Assessment": occ_assessment[c][0],
     "Evidence note": occ_assessment[c][1]}
    for c in occ_diff_codes
])
print(occ_review.to_string(index=False))
print("\nAssessment counts:", occ_review["Assessment"].value_counts().to_dict())
print("Decision: compare OCCP codes directly for the shared 2018 coding vintage, with code 6800 flagged for sensitivity analysis until its PUMS aggregation is confirmed.")


Canonical OCCP labels loaded from cached Census definition.csv for 2018; both year dictionaries validated.
 OCCP                                                                             2018 description                                                                    2018 canonical PUMS label                                                                                              2021 description   Assessment                                                                                                Evidence note
 1555                             Other Engineering Technologists And Technicians, Except Drafters                             Other Engineering Technologists And Technicians, Except Drafters                                          ENG-Other Engineering Technologists And Technicians, Except Drafters Wording only                                                       Category title/prefix formatting; same 2018 OCCP code.
 2636                                        

## Frequency of the unresolved code

Count code 6800 after the same `adult_filter` criteria used by ACSIncome. The summary covers all 50 states and the project's existing six-state evaluation set. It reports unweighted record counts and the share of filtered records; it does not infer the occupation composition inside code 6800.


In [13]:
from folktables.load_acs import _STATE_CODES

evaluation_states = {"MS", "WV", "NM", "AR", "LA", "MT"}
occ6800_summary = []
for year in (2018, 2021):
    us_n = us_code_n = eval_n = eval_code_n = 0
    state_counts = {}
    for state, fips in _STATE_CODES.items():
        path = DATA_DIR / str(year) / "1-Year" / f"psam_p{fips}.csv"
        if not path.is_file():
            continue
        state_n = state_code_n = 0
        for chunk in pd.read_csv(
            path, usecols=["AGEP", "PINCP", "WKHP", "PWGTP", "OCCP"],
            dtype={"OCCP": "string"}, chunksize=250_000,
        ):
            keep = ((chunk["AGEP"] > 16) & (chunk["PINCP"] > 100)
                    & (chunk["WKHP"] > 0) & (chunk["PWGTP"] >= 1))
            state_n += int(keep.sum())
            state_code_n += int(((chunk["OCCP"] == "6800") & keep).sum())
        us_n += state_n; us_code_n += state_code_n
        state_counts[state] = state_code_n
        if state in evaluation_states:
            eval_n += state_n; eval_code_n += state_code_n
    occ6800_summary.append({
        "year": year, "US_filtered_rows": us_n, "US_code_6800_rows": us_code_n,
        "US_share_pct": 100 * us_code_n / us_n,
        "evaluation_filtered_rows": eval_n,
        "evaluation_code_6800_rows": eval_code_n,
        "evaluation_share_pct": 100 * eval_code_n / eval_n,
    })
    print(year, "evaluation code-6800 by state:",
          {s: state_counts[s] for s in sorted(evaluation_states)})
print(pd.DataFrame(occ6800_summary).round(4).to_string(index=False))


2018 evaluation code-6800 by state: {'AR': 4, 'LA': 33, 'MS': 11, 'MT': 4, 'NM': 8, 'WV': 8}
2021 evaluation code-6800 by state: {'AR': 4, 'LA': 15, 'MS': 10, 'MT': 4, 'NM': 5, 'WV': 4}
 year  US_filtered_rows  US_code_6800_rows  US_share_pct  evaluation_filtered_rows  evaluation_code_6800_rows  evaluation_share_pct
 2018           1655429                394        0.0238                     70062                         68                0.0971
 2021           1621285                233        0.0144                     67362                         42                0.0623


## Reassess matching with numerical coarsening

Use the same 1,000 California 2018 rows and the full filtered California 2018 public reference for every representation. Original and jittered queries use exact categorical meanings and an absolute tolerance of 1 on both numeric fields. Coarsened queries use exact categorical meanings and require the reference record to fall in the same age and work-hour bands; within that candidate set, the attacker ranks raw public values by distance to the released band midpoints. The matcher knows the category map and coarsening/jitter rules, but not the random jitter draws.

Coarsening rule: AGEP is mapped to 5-year bands from 17 upward and represented by the band midpoint. WKHP is rounded to the nearest 5 hours and clipped to 1–98. Top-coded 99 remains a separate value. The same deterministic rules are applied to 2018 and 2021. The same rule is applied to California 2021 as a cross-year consistency check. No person identifiers are used.


In [15]:
# Load only public reference features; no person identifiers are used.
public_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person", root_dir=str(DATA_DIR))
public_raw = public_source.get_data(states=["CA"], download=False)
public_filtered = adult_filter(harmonize_features(public_raw, 2018))
public_X = np.nan_to_num(public_filtered[features].to_numpy())
public_encoded = encode_categories(public_X)
del public_raw, public_filtered

cat_cols = [features.index(f) for f in categorical_features]
num_cols = [features.index("AGEP"), features.index("WKHP")]
reference_groups = {}
for row in public_encoded:
    key = tuple(int(row[c]) for c in cat_cols)
    reference_groups.setdefault(key, []).append(row[num_cols])
reference_groups = {k: np.asarray(v) for k, v in reference_groups.items()}

def match_counts(candidate):
    exact_counts, near_counts = {0.0:[], 0.25:[], 0.5:[], 1.0:[]}, {0.25:[], 0.5:[], 1.0:[]}
    for row in candidate:
        key = tuple(int(row[c]) for c in cat_cols)
        refs = reference_groups.get(key, np.empty((0, 2)))
        distances = np.abs(refs - row[num_cols])
        exact_counts[0.0].append(int(np.all(distances == 0, axis=1).sum()))
        for tolerance in near_counts:
            near_counts[tolerance].append(int(np.all(distances <= tolerance, axis=1).sum()))
    return exact_counts[0.0], near_counts

def report_match(name, candidate):
    exact, near = match_counts(candidate)
    result = {"candidate":name,"exact_rows_with_match_pct":100*np.mean(np.array(exact)>0),"exact_unique_match_pct":100*np.mean(np.array(exact)==1)}
    for tolerance, counts in near.items():
        a=np.asarray(counts)
        result[f"within_{tolerance:g}_match_pct"] = 100*np.mean(a>0)
        result[f"within_{tolerance:g}_median_matches"] = float(np.median(a))
        result[f"within_{tolerance:g}_unique_pct"] = 100*np.mean(a==1)
    return result

match_rows = [report_match("original", X_categorical)]
combined_checks = []
for magnitude in magnitudes:
    combined = X_categorical.copy()
    for feature in ("AGEP", "WKHP"):
        combined[:, features.index(feature)] = jitter_candidates[(feature, magnitude)]
    shuffled_order = np.random.default_rng(42).permutation(len(y))
    combined_s = combined[shuffled_order]
    y_s, states_s = y[shuffled_order], state_ids[shuffled_order]
    inv = np.argsort(shuffled_order)
    aligned = (np.array_equal(combined_s[inv], combined)
               and np.array_equal(y_s[inv], y)
               and np.array_equal(states_s[inv], state_ids))
    decoded = decode_categories(combined)
    cats_valid = all(np.isfinite(combined[:, c]).all() for c in cat_cols)
    categories_recovered = np.array_equal(decoded[:, cat_cols], X[:, cat_cols])
    age_delta = np.abs(combined[:, features.index("AGEP")] - X[:, features.index("AGEP")])
    wkhp_delta = np.abs(combined[:, features.index("WKHP")] - X[:, features.index("WKHP")])
    valid_num = (np.all((combined[:, features.index("AGEP")] >= 17) & (combined[:, features.index("AGEP")] <= 99))
                 and np.all((combined[:, features.index("WKHP")] >= 1) & (combined[:, features.index("WKHP")] <= 99)))
    labels_same = np.array_equal(y, y_original)
    state_same = np.array_equal(state_ids, state_ids_original)
    order_same = features == ["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP", "SEX", "RAC1P"]
    combined_checks.append({"half_width":magnitude,"rows_aligned":aligned,"labels_same":labels_same,"state_ids_same":state_same,"categoricals_valid":cats_valid,"category_round_trip":categories_recovered,"feature_order_same":order_same,"numeric_bounds_valid":valid_num,"max_AGEP_change":age_delta.max(),"max_WKHP_change":wkhp_delta.max(),"SCHL_order_sidecar":len(schl_order_values)==len(y)})
    match_rows.append(report_match(f"jitter_{magnitude:g}", combined))

print("Combined integrity by candidate:")
print(pd.DataFrame(combined_checks).round(4).to_string(index=False))
print("Public-reference matching (per-row feature matches):")
print(pd.DataFrame(match_rows).round(3).to_string(index=False))
assert all(all(row[k] for k in ["rows_aligned","labels_same","state_ids_same","categoricals_valid","category_round_trip","feature_order_same","numeric_bounds_valid","SCHL_order_sidecar"]) for row in combined_checks)
print("No packaged archive was written or changed.")

Combined integrity by candidate:
 half_width  rows_aligned  labels_same  state_ids_same  categoricals_valid  category_round_trip  feature_order_same  numeric_bounds_valid  max_AGEP_change  max_WKHP_change  SCHL_order_sidecar
       0.10          True         True            True                True                 True                True                  True           0.1000           0.0999                True
       0.25          True         True            True                True                 True                True                  True           0.2497           0.2493                True
       0.50          True         True            True                True                 True                True                  True           0.5000           0.4998                True
       1.00          True         True            True                True                 True                True                  True           0.9998           0.9979                True
Public-

## Limits and next checks

A reversible category map preserves category information and cannot by itself prevent linkage. The exact and approximate rates above use a specific public 2018 California reference, leave all categories exact, and are separate metrics. They do not establish privacy against other tables or records. No final jitter magnitude is selected here. Do not regenerate the four packaged archives until the mapping, utility, and broader linkage audit are independently accepted.

## Linkage ambiguity and source-row recovery

This diagnostic retains only a temporary row-position correspondence between the 1,000-row source sample and transformed candidates. It never uses or writes Census person identifiers. The public reference is the same California 2018 file used above, so the result measures recovery within that reference only; it does not test false candidates from other records or years.

For categorical recoding, matching must compare meanings: decode both the transformed sample and public reference to canonical source codes before candidate generation.


In [18]:
# Confirm source-row alignment to the full public reference, then evaluate four representations.
assert len(public_X) == 195665
assert np.array_equal(public_X[:len(X)], X), "Sample/reference row correspondence failed."
source_ref_index = np.arange(len(X))
public_canonical = decode_categories(public_encoded)
public_cats = public_canonical[:, cat_cols]
public_nums = public_canonical[:, num_cols]

# Deterministic, year-independent midpoint coarsening for numeric features only.
def coarsen_values(values, feature):
    values = np.asarray(values, dtype=float)
    result = values.copy()
    if feature == "AGEP":
        ordinary = values < 99
        lower = 17 + 5 * np.floor((values[ordinary] - 17) / 5)
        upper = np.minimum(lower + 4, 98)
    elif feature == "WKHP":
        ordinary = values < 99
        result[ordinary] = np.clip(np.floor(values[ordinary] / 5 + 0.5) * 5, 1, 98)
        return result
    else:
        raise ValueError(feature)
    result[ordinary] = (lower + upper) / 2
    return result

def coarsen_matrix(array):
    result = np.asarray(array).copy()
    for feature in ("AGEP", "WKHP"):
        col = features.index(feature)
        result[:, col] = coarsen_values(result[:, col], feature)
    return result

def joint_shuffle_check(array, source_index):
    order = np.random.default_rng(42).permutation(len(y))
    arr_s, y_s, state_s, source_s = array[order], y[order], state_ids[order], source_index[order]
    inv = np.argsort(order)
    aligned = (np.array_equal(arr_s[inv], array)
               and np.array_equal(y_s[inv], y)
               and np.array_equal(state_s[inv], state_ids)
               and np.array_equal(source_s[inv], source_index))
    return arr_s, source_s, aligned, np.array_equal(y_s[inv], y)

def match_metrics(candidate, source_index, *, category_map_known=False, coarse=False, tolerance=1.0):
    canonical = decode_categories(candidate) if category_map_known else candidate
    cats = canonical[:, cat_cols]
    nums = canonical[:, num_cols]
    counts, source_hit, top1, nearest_ties = [], [], [], []
    public_coarse = np.column_stack([
        coarsen_values(public_nums[:, 0], "AGEP"),
        coarsen_values(public_nums[:, 1], "WKHP"),
    ]) if coarse else None
    for cats_q, nums_q, source_idx in zip(cats, nums, source_index):
        cat_ok = np.all(public_cats == cats_q, axis=1)
        if coarse:
            eligible = cat_ok & np.all(public_coarse == nums_q, axis=1)
        else:
            eligible = cat_ok & np.all(np.abs(public_nums - nums_q) <= tolerance, axis=1)
        ids = np.flatnonzero(eligible)
        counts.append(len(ids))
        source_hit.append(bool(np.any(ids == source_idx)))
        if len(ids):
            # Rank candidates by distance from their raw public values to the released numeric query.
            dist = np.sqrt(np.sum((public_nums[ids] - nums_q) ** 2, axis=1))
            nearest = ids[np.isclose(dist, dist.min(), rtol=0, atol=1e-12)]
            top1.append(1 / len(nearest) if source_idx in nearest else 0.0)
            nearest_ties.append(len(nearest) > 1)
        else:
            top1.append(0.0); nearest_ties.append(False)
    counts = np.asarray(counts)
    return {
        "unique_match_pct": 100 * np.mean(counts == 1),
        "median_candidates": float(np.median(counts)),
        "mean_candidates": float(np.mean(counts)),
        "source_recall_pct": 100 * np.mean(source_hit),
        "top1_tie_adjusted_pct": 100 * np.mean(top1),
        "nearest_tie_pct": 100 * np.mean(nearest_ties),
        "candidate_coverage_pct": 100 * np.mean(counts > 0),
    }

# Representation 1: original values. Representation 2: existing largest tested jitter + reversible map.
jitter_full = X_categorical.copy()
for feature in ("AGEP", "WKHP"):
    jitter_full[:, features.index(feature)] = jitter_candidates[(feature, 1.0)]

# Representation 3: coarsening only. Representation 4: same coarsening plus existing category map.
coarse_raw = coarsen_matrix(X)
coarse_mapped = coarsen_matrix(X_categorical)

# Jointly permute transformed features, labels, state IDs, and temporary source-row indices.
jitter_s, jitter_src, jitter_aligned, jitter_labels_ok = joint_shuffle_check(jitter_full, source_ref_index)
coarse_s, coarse_src, coarse_aligned, coarse_labels_ok = joint_shuffle_check(coarse_raw, source_ref_index)
coarse_map_s, coarse_map_src, coarse_map_aligned, coarse_map_labels_ok = joint_shuffle_check(coarse_mapped, source_ref_index)
assert all([jitter_aligned, jitter_labels_ok, coarse_aligned, coarse_labels_ok,
            coarse_map_aligned, coarse_map_labels_ok])
assert np.array_equal(y, y_original)

match_results = [
    {"representation":"Original", "rule":"exact categories; +/-1 numeric", **match_metrics(X, source_ref_index)},
    {"representation":"Jitter + category map", "rule":"decode map; +/-1 numeric", **match_metrics(jitter_s, jitter_src, category_map_known=True)},
    {"representation":"Coarsened numerics", "rule":"exact category; exact same bands", **match_metrics(coarse_s, coarse_src, coarse=True)},
    {"representation":"Coarsening + category map", "rule":"decode map; exact same bands", **match_metrics(coarse_map_s, coarse_map_src, category_map_known=True, coarse=True)},
]
print("Reference rows:", len(public_X), "| sample rows:", len(X), "| transformation rules known to matcher: yes")
print(pd.DataFrame(match_results).round(3).to_string(index=False))

# Distortion and information retained in numerical fields.
from scipy.stats import spearmanr
quality_rows = []
for feature in ("AGEP", "WKHP"):
    col = features.index(feature); orig = X[:, col]; transformed = coarse_raw[:, col]
    quality_rows.append({"feature":feature, "mean_abs_change":np.mean(np.abs(transformed-orig)),
                         "max_abs_change":np.max(np.abs(transformed-orig)),
                         "mean_change":np.mean(transformed-orig),
                         "std_change":np.std(transformed)-np.std(orig),
                         "spearman_rank":spearmanr(orig, transformed).statistic})
print("Coarsening distortion on 1,000 rows:")
print(pd.DataFrame(quality_rows).round(4).to_string(index=False))
print("Joint shuffle alignment errors: 0; labels preserved: yes; feature order preserved:", features == ["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP", "SEX", "RAC1P"])

# Apply the identical, year-independent bin functions to visible California 2021 features only.
source_2021 = ACSDataSource(survey_year="2021", horizon="1-Year", survey="person", root_dir=str(DATA_DIR))
raw_2021 = source_2021.get_data(states=["CA"], download=False)
filtered_2021 = adult_filter(harmonize_features(raw_2021, 2021))
X_ca_2021 = np.nan_to_num(filtered_2021[features].to_numpy())
X_ca_2021_coarse = coarsen_matrix(X_ca_2021)
for feature in ("AGEP", "WKHP"):
    col = features.index(feature)
    assert np.array_equal(X_ca_2021[X_ca_2021[:, col] == 99, col], X_ca_2021_coarse[X_ca_2021[:, col] == 99, col])
print("Cross-year check: same fixed bins applied to", len(X_ca_2021), "California 2021 rows; top-code 99 preserved.")
print("No packaged archive was written or changed.")


Reference rows: 195665 | sample rows: 1000 | transformation rules known to matcher: yes
           representation                             rule  unique_match_pct  median_candidates  mean_candidates  source_recall_pct  top1_tie_adjusted_pct  nearest_tie_pct  candidate_coverage_pct
                 Original   exact categories; +/-1 numeric              25.4                2.0            2.660              100.0                 53.124             73.8                   100.0
    Jitter + category map         decode map; +/-1 numeric              25.7                2.0            2.644              100.0                 52.899             73.5                   100.0
       Coarsened numerics exact category; exact same bands              25.2                2.0            2.690              100.0                 51.624             73.1                   100.0
Coarsening + category map     decode map; exact same bands              25.2                2.0            2.690              10

## Matching audit interpretation

All four representations use the same 1,000 California 2018 sample and the same 195,665-row California 2018 reference. The feature-only reference rows are in the same order as the source sample; temporary row indices provide source ground truth without using person identifiers. Categorical codes are matched by their decoded meanings. The matcher knows the reversible category mapping and stated numerical transformation rules.

The coarsening metrics use exact agreement on the released numerical bands; original and jittered metrics use exact categories and the same +/-1 numeric tolerance. Thus these candidate sets reflect attacker-aware rules for each published representation. Compare source recall and top-1 alongside ambiguity: lower recall alone is not a privacy result. These remain diagnostics against one reference population, not a formal privacy guarantee.

No model utility comparison is included in this checkpoint. Do not overwrite packaged archives based on matching results alone.


## Matching audit — conclusion

| Finding | Result |
|---|---:|
| Original top-1 matching accuracy | 53.12% |
| Best transformed top-1 accuracy | 51.62% |
| Original unique-match rate | 25.4% |
| Transformed unique-match rate | 25.2% |
| True-source recall | 100% in both |
| Decision | Protection remains insufficient; no final transformation approved |

`OCCP=6800` remains a documented cross-year label/aggregation uncertainty. No packaged archives were changed.
